# IRI Standards Agent — Production v1.0 Test
Validates agent against **Policy Inquiry v1.1.0** spec + Data Dictionary.

In [0]:
%pip install strands-agents strands-agents-tools openai pyyaml databricks-sdk openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys
sys.path.insert(0, '/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent')

from agent.agent_prod import create_agent
agent = create_agent()

# Patch: Databricks endpoints don't always return token usage metrics,
# which causes strands event loop to crash with NoneType += error
_original_update = agent.event_loop_metrics.update_usage
def _safe_update(usage):
    try:
        _original_update(usage)
    except TypeError:
        pass  # Skip metrics update when usage data is None
agent.event_loop_metrics.update_usage = _safe_update
print("[patch] Applied metrics safety patch for Databricks endpoint")

In [0]:
# Policy Inquiry v1.1.3 — load corrected spec from local file
spec_path = '/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output/PolicyInquiry_v1.1.3.yml'
dd_url = "https://raw.githubusercontent.com/Insured-Retirement-Institute/Policy-Inquiry/edwardmruiz-2/Data_Dictionary_PolicyInquiry_V1.1.0.xlsx"

with open(spec_path) as f:
    spec_yaml = f.read()

print(f"Loaded spec: {spec_path}")
print(f"Size: {len(spec_yaml):,} chars, {spec_yaml.count(chr(10))+1} lines")
print(f"DD URL: {dd_url}")

In [0]:
review_prompt = f"""Review this IRI Digital-First OpenAPI 3.1 YAML specification for Policy Inquiry v1.1.3.

This is a REVISION of a previously reviewed spec. Critical issues have been resolved in earlier revisions.
This version specifically addresses remaining governance deductions from v1.1.2:
- Data Dictionary alignment for PartyRole and PartyStatus enums
- Removal of UnderlyingAssets.transactionId response echo
- Rider example alignment after participantRoleCode rename
- Additional constraints/documentation for classification and numeric-string fields

Also fetch and cross-check the Data Dictionary:
- Data Dictionary URL: {dd_url}

Perform a full review including:
- Structural validation (OpenAPI 3.1 compliance)
- Style guide enforcement (IRI conventions)
- Cross-spec consistency against published IRI specs
- Data Dictionary alignment (verify DD fields match schema definitions)

Provide:
1. The full governance scorecard with category scores
2. A list of ALL findings (Critical, Moderate, Minor) with evidence
3. Any mismatches between the YAML spec and the Data Dictionary
4. Whether this revision reaches 100/100

Here is the spec:

```yaml
{spec_yaml}
```"""

result = agent(review_prompt)
review_output = str(result)
print(review_output[:5000])

In [0]:
import requests
import yaml
import copy

# Fetch the original v1.1.0 patched spec from GitHub
original_url = "https://raw.githubusercontent.com/Insured-Retirement-Institute/Policy-Inquiry/edwardmruiz-2/Policy_Inquiry_V1.1.0_patched.yaml"
resp = requests.get(original_url)
resp.raise_for_status()
raw_yaml = resp.text
print(f"Fetched spec: {len(raw_yaml)} chars, {raw_yaml.count(chr(10))+1} lines")

# Parse
spec = yaml.safe_load(raw_yaml)
print(f"OpenAPI version: {spec.get('openapi')}")
print(f"Title: {spec['info']['title']}")
print(f"Version: {spec['info']['version']}")
print(f"Schemas defined: {len(spec['components']['schemas'])}")
print(f"Paths defined: {len(spec['paths'])}")

In [0]:
# ============================================================
# Apply Critical Fixes
# ============================================================

schemas = spec['components']['schemas']

# --- C-1: Delete orphan/shadow schemas ---
orphan_schemas = [
    'AccountValues', 'LoanValues', 'WithdrawalValues', 'MarketValueAdjustment',
    'RmdInfo', 'AuthorizationTransaction', 'Fund', 'RateTier', 'FundSegment',
    'RateLockInfo', 'TransferRestrictInfo', 'PolicyDate', 'IncomeArrangement',
    'Annuitization', 'PeriodCertain', 'SurvivorReduction', 'IncomePayment',
    'PaymentAmount', 'PaymentAdjustments', 'TaxExclusion', 'UnderlyingAsset',
    'SystematicProgramParty', 'TaxWithholdingParty', 'RiderParticipant',
    'RiderCharge', 'Transaction', 'TransactionExternalIdentifier',
    'TransactionCharge', 'RestrictionReasonDocEntry', 'RestrictionReasonCatalog'
]
removed_orphans = 0
for name in orphan_schemas:
    if name in schemas:
        del schemas[name]
        removed_orphans += 1
print(f"C-1: Removed {removed_orphans} orphan schemas")

# --- C-2: Remove policyNumber + associatedFirmId from response body schemas ---
response_schemas = [
    'PolicySummary', 'PolicyValue', 'PolicyFunds', 'PartiesList',
    'PolicyProducers', 'PolicyDates', 'SystematicProgramsList',
    'PayoutsList', 'TransactionsList', 'RidersList', 'UnderlyingAssets'
]
c2_fixes = 0
for schema_name in response_schemas:
    if schema_name not in schemas:
        continue
    s = schemas[schema_name]
    props = s.get('properties', {})
    for field in ['policyNumber', 'associatedFirmId']:
        if field in props:
            del props[field]
            c2_fixes += 1
    if 'required' in s:
        s['required'] = [r for r in s['required'] if r not in ('policyNumber', 'associatedFirmId')]
        if not s['required']:
            del s['required']
print(f"C-2: Removed {c2_fixes} policyNumber/associatedFirmId fields from response schemas")

# --- C-3: Clean TransactionsList leaked parameter-echo block ---
if 'TransactionsList' in schemas:
    tl = schemas['TransactionsList']
    tl_props = tl.get('properties', {})
    legit_fields = {'startIndex', 'itemsCount', 'totalItemsCount', 'transactions'}
    leaked = [k for k in tl_props if k not in legit_fields]
    for field in leaked:
        del tl_props[field]
    print(f"C-3: Removed {len(leaked)} leaked fields from TransactionsList")

# --- C-4: Fix TaxWithholdingInstruction if/then guard ---
if 'TaxWithholdingInstruction' in schemas:
    twi = schemas['TaxWithholdingInstruction']
    if 'allOf' in twi:
        for item in twi['allOf']:
            if 'if' in item:
                if_block = item['if']
                if 'properties' in if_block and 'taxWithholdingType' in if_block.get('properties', {}):
                    if 'required' not in if_block:
                        if_block['required'] = ['taxWithholdingType']
                        print("C-4: Added required: [taxWithholdingType] to if-block")

# --- C-6: Constrain npn field ---
if 'ProducerExternalIds' in schemas:
    pei = schemas['ProducerExternalIds']
    npn_prop = pei.get('properties', {}).get('npn', {})
    if npn_prop:
        npn_prop['pattern'] = '^[0-9]{1,10}$'
        npn_prop['maxLength'] = 10
        print(f"C-6: Constrained npn")

print(f"\nSchemas remaining after Critical fixes: {len(schemas)}")

In [0]:
# ============================================================
# Apply Moderate and Minor Fixes (Round 2 + Round 3)
# ============================================================

# --- M-3: Move pagination fields from SystematicProgram to SystematicProgramsList ---
if 'SystematicProgram' in schemas and 'SystematicProgramsList' in schemas:
    sp = schemas['SystematicProgram']
    spl = schemas['SystematicProgramsList']
    pagination_fields = ['startIndex', 'itemsCount', 'totalItemsCount']
    sp_props = sp.get('properties', {})
    spl_props = spl.get('properties', {})
    moved = 0
    for field in pagination_fields:
        if field in sp_props:
            if field not in spl_props:
                spl_props[field] = sp_props[field]
            del sp_props[field]
            moved += 1
    print(f"M-3: Moved {moved} pagination fields from SystematicProgram to SystematicProgramsList")

# --- M-3b: Add pagination fields to PolicyProducers ---
if 'PolicyProducers' in schemas:
    pp_props = schemas['PolicyProducers'].get('properties', {})
    if 'startIndex' not in pp_props:
        pp_props['startIndex'] = {'type': 'integer', 'minimum': 0, 'description': 'Zero-based index of the first item returned.'}
        pp_props['itemsCount'] = {'type': 'integer', 'minimum': 0, 'description': 'Number of items in the current response.'}
        pp_props['totalItemsCount'] = {'type': 'integer', 'minimum': 0, 'description': 'Total number of items available.'}
        print("M-3b: Added pagination fields to PolicyProducers")

# --- M-4: Remove npn from ProducerExternalIds required (too strict for read API) ---
if 'ProducerExternalIds' in schemas:
    pei = schemas['ProducerExternalIds']
    if 'required' in pei and 'npn' in pei['required']:
        pei['required'].remove('npn')
        if not pei['required']:
            del pei['required']
        print("M-4: Removed npn from ProducerExternalIds.required")

# --- M-5: Remove partyStatus/partyRole from Party body schema ---
if 'Party' in schemas:
    party_props = schemas['Party'].get('properties', {})
    removed_party = 0
    for field in ['partyStatus', 'partyRole']:
        if field in party_props:
            del party_props[field]
            removed_party += 1
    if 'required' in schemas['Party']:
        schemas['Party']['required'] = [
            r for r in schemas['Party']['required']
            if r not in ('allocationPercentage', 'paymentForm')
        ]
        if not schemas['Party']['required']:
            del schemas['Party']['required']
    print(f"M-5: Removed {removed_party} stray fields from Party; relaxed required")

# --- M-5b: Fix issueCountry enum — replace YAML boolean false with 'NO' ---
def fix_issue_country_enum(schema_dict):
    if isinstance(schema_dict, dict):
        for key, val in schema_dict.items():
            if key == 'issueCountry' and isinstance(val, dict):
                enum_list = val.get('enum', [])
                if False in enum_list:
                    idx = enum_list.index(False)
                    enum_list[idx] = 'NO'
                    print(f"M-5b: Fixed issueCountry enum — replaced YAML 'false' with 'NO' (Norway)")
                # Deduplicate
                seen = set()
                deduped = []
                for item in enum_list:
                    if item not in seen:
                        seen.add(item)
                        deduped.append(item)
                if len(deduped) < len(enum_list):
                    print(f"m-5: Deduplicated issueCountry enum ({len(enum_list)} -> {len(deduped)})")
                    val['enum'] = deduped
                return
            elif isinstance(val, (dict, list)):
                fix_issue_country_enum(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fix_issue_country_enum(item)

fix_issue_country_enum(schemas)

# --- M-6: Add pattern to all taxId fields ---
def add_taxid_pattern(schema_dict, path=""):
    fixes = 0
    if isinstance(schema_dict, dict):
        for key, val in schema_dict.items():
            if key == 'taxId' and isinstance(val, dict) and val.get('type') == 'string':
                if 'pattern' not in val:
                    val['pattern'] = '^[0-9]{9}$'
                    val['minLength'] = 9
                    fixes += 1
            elif isinstance(val, (dict, list)):
                fixes += add_taxid_pattern(val, f"{path}.{key}")
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += add_taxid_pattern(item, path)
    return fixes

taxid_fixes = add_taxid_pattern(schemas)
print(f"M-6: Added pattern to {taxid_fixes} taxId fields")

# --- M-12: Add min/max to riderBenefitPercent ---
if 'Rider' in schemas:
    rider_props = schemas['Rider'].get('properties', {})
    rbp = rider_props.get('riderBenefitPercent', {})
    if rbp and 'minimum' not in rbp:
        rbp['minimum'] = 0
        rbp['maximum'] = 100
        print("M-12: Added min/max to riderBenefitPercent")

# --- M-13: Fix numOfModalOccurrences type ---
if 'SystematicProgram' in schemas:
    sp_props = schemas['SystematicProgram'].get('properties', {})
    nomo = sp_props.get('numOfModalOccurrences', {})
    if nomo and nomo.get('type') == 'number':
        nomo['type'] = 'integer'
        nomo['minimum'] = 0
        if 'maximum' in nomo and nomo['maximum'] == 9999999999.99:
            nomo['maximum'] = 9999999999
        print("M-13: Changed numOfModalOccurrences to integer")

# --- M-14: Fix incomePayments singular object -> rename to incomePayment ---
def fix_income_payments(schema_dict):
    fixes = 0
    if isinstance(schema_dict, dict):
        props = schema_dict.get('properties', {})
        if 'incomePayments' in props:
            val = props['incomePayments']
            if isinstance(val, dict) and val.get('type') == 'object':
                props['incomePayment'] = props.pop('incomePayments')
                fixes += 1
        if 'required' in schema_dict:
            req = schema_dict['required']
            if isinstance(req, list) and 'incomePayments' in req:
                idx = req.index('incomePayments')
                req[idx] = 'incomePayment'
                print("  M-9: Fixed required reference incomePayments -> incomePayment")
        for key, val in list(schema_dict.items()):
            if isinstance(val, (dict, list)):
                fixes += fix_income_payments(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += fix_income_payments(item)
    return fixes

ip_fixes = fix_income_payments(schemas)
print(f"M-14: Renamed {ip_fixes} incomePayments -> incomePayment")

# --- M-1/M-2: Fix administrativeTransaction rename follow-through ---
def fix_admin_transaction(schema_dict):
    fixes = 0
    if isinstance(schema_dict, dict):
        props = schema_dict.get('properties', {})
        if 'administrativeTransaction' in props:
            val = props['administrativeTransaction']
            if isinstance(val, dict) and val.get('type') == 'array':
                props['administrativeTransactions'] = props.pop('administrativeTransaction')
                fixes += 1
        if 'required' in schema_dict:
            req = schema_dict['required']
            if isinstance(req, list) and 'administrativeTransaction' in req:
                idx = req.index('administrativeTransaction')
                req[idx] = 'administrativeTransactions'
                print("  M-1: Fixed required reference administrativeTransaction -> administrativeTransactions")
        for val in list(schema_dict.values()):
            if isinstance(val, (dict, list)):
                fixes += fix_admin_transaction(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += fix_admin_transaction(item)
    return fixes

at_fix = fix_admin_transaction(schemas)
print(f"m-13/M-1: Renamed {at_fix} administrativeTransaction -> administrativeTransactions (with required fix)")

# --- M-2: Fix examples that use old key names ---
def fix_examples(spec_dict):
    fixes = 0
    if isinstance(spec_dict, dict):
        if 'example' in spec_dict and isinstance(spec_dict['example'], dict):
            fixes += fix_example_keys(spec_dict['example'])
        if 'examples' in spec_dict and isinstance(spec_dict['examples'], dict):
            for ex_name, ex_val in spec_dict['examples'].items():
                if isinstance(ex_val, dict) and 'value' in ex_val:
                    fixes += fix_example_keys(ex_val['value'])
        for val in spec_dict.values():
            if isinstance(val, (dict, list)):
                fixes += fix_examples(val)
    elif isinstance(spec_dict, list):
        for item in spec_dict:
            fixes += fix_examples(item)
    return fixes

def fix_example_keys(obj):
    fixes = 0
    if isinstance(obj, dict):
        if 'administrativeTransaction' in obj:
            obj['administrativeTransactions'] = obj.pop('administrativeTransaction')
            fixes += 1
        if 'incomePayments' in obj:
            obj['incomePayment'] = obj.pop('incomePayments')
            fixes += 1
        for val in list(obj.values()):
            if isinstance(val, (dict, list)):
                fixes += fix_example_keys(val)
    elif isinstance(obj, list):
        for item in obj:
            fixes += fix_example_keys(item)
    return fixes

ex_fixes = fix_examples(spec)
print(f"M-2: Fixed {ex_fixes} example keys (renamed to match schema)")

# --- m-2: Remove correlationId from Error body ---
if 'Error' in schemas:
    err_props = schemas['Error'].get('properties', {})
    if 'correlationId' in err_props:
        del err_props['correlationId']
        print("m-2: Removed correlationId from Error body")

# --- DD-1: Tighten firstName/middleName/lastName maxLength from 105 to 100 ---
def fix_name_lengths(schema_dict):
    fixes = 0
    if isinstance(schema_dict, dict):
        for key, val in schema_dict.items():
            if key in ('firstName', 'middleName', 'lastName') and isinstance(val, dict):
                if val.get('maxLength') == 105:
                    val['maxLength'] = 100
                    fixes += 1
            elif isinstance(val, (dict, list)):
                fixes += fix_name_lengths(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += fix_name_lengths(item)
    return fixes

name_fixes = fix_name_lengths(schemas)
print(f"DD-1: Tightened {name_fixes} name fields maxLength 105 -> 100 (DD alignment)")

# --- M-7 / R2-M-2: Fix conditional field descriptions (use correct enum values) ---
if 'SystematicProgram' in schemas:
    sp_props = schemas['SystematicProgram'].get('properties', {})
    if 'requestedAmount' in sp_props:
        ra = sp_props['requestedAmount']
        if isinstance(ra, dict):
            ra['description'] = "Required when amountType is 'AMOUNT'. Mutually exclusive with requestedPercentage."
    if 'requestedPercentage' in sp_props:
        rp = sp_props['requestedPercentage']
        if isinstance(rp, dict):
            rp['description'] = "Required when amountType is 'PERCENTAGE'. Mutually exclusive with requestedAmount."
    print("R2-M-2: Fixed requestedAmount/requestedPercentage descriptions (AMOUNT not DOLLAR)")

# --- R2-M-3: Fix name field pattern quantifiers {1,105} -> {1,100} ---
import re
def fix_name_patterns(schema_dict):
    fixes = 0
    if isinstance(schema_dict, dict):
        for key, val in schema_dict.items():
            if key in ('firstName', 'middleName', 'lastName', 'name') and isinstance(val, dict):
                pattern = val.get('pattern', '')
                if '{1,105}' in pattern:
                    val['pattern'] = pattern.replace('{1,105}', '{1,100}')
                    fixes += 1
            elif isinstance(val, (dict, list)):
                fixes += fix_name_patterns(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += fix_name_patterns(item)
    return fixes

pattern_fixes = fix_name_patterns(schemas)
print(f"R2-M-3: Fixed {pattern_fixes} name field pattern quantifiers {{1,105}} -> {{1,100}}")

# --- R2-M-4: Add inner required fields to rateLockInfo ---
def fix_rate_lock_info(schema_dict):
    fixes = 0
    if isinstance(schema_dict, dict):
        for key, val in schema_dict.items():
            if key == 'rateLockInfo' and isinstance(val, dict) and val.get('type') == 'object':
                props = val.get('properties', {})
                if props and 'required' not in val:
                    # rateType is the minimum meaningful field when lock is enabled
                    inner_required = []
                    if 'rateType' in props:
                        inner_required.append('rateType')
                    if 'lockStartDate' in props:
                        inner_required.append('lockStartDate')
                    if 'lockEndDate' in props:
                        inner_required.append('lockEndDate')
                    if inner_required:
                        val['required'] = inner_required
                        fixes += 1
            elif isinstance(val, (dict, list)):
                fixes += fix_rate_lock_info(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += fix_rate_lock_info(item)
    return fixes

rl_fixes = fix_rate_lock_info(schemas)
print(f"R2-M-4: Added inner required fields to {rl_fixes} rateLockInfo object(s)")

# --- R2-M-1: Align assetClass enums + constrain insuranceProductSubAccountAssetClassCode ---
# Expand assetClass to include EQUITY (seen in examples) and constrain the other field
asset_class_enum = ['FIXED', 'VARIABLE', 'INDEXED', 'MODEL', 'CASH', 'EQUITY', 'OTHER']
def fix_asset_class(schema_dict):
    fixes = 0
    if isinstance(schema_dict, dict):
        for key, val in schema_dict.items():
            if key == 'assetClass' and isinstance(val, dict) and 'enum' in val:
                val['enum'] = asset_class_enum
                fixes += 1
            elif key == 'insuranceProductSubAccountAssetClassCode' and isinstance(val, dict):
                # Constrain to same enum instead of free-form pattern
                val['enum'] = asset_class_enum
                val.pop('pattern', None)
                val['description'] = val.get('description', '') or 'Asset class classification code.'
                fixes += 1
            elif isinstance(val, (dict, list)):
                fixes += fix_asset_class(val)
    elif isinstance(schema_dict, list):
        for item in schema_dict:
            fixes += fix_asset_class(item)
    return fixes

ac_fixes = fix_asset_class(schemas)
print(f"R2-M-1: Aligned {ac_fixes} assetClass/insuranceProductSubAccountAssetClassCode fields to shared enum")

print(f"\n\u2705 All fixes applied (Round 3). Final schema count: {len(schemas)}")

In [0]:
# ============================================================
# Apply Round 2 Minor Fixes → v1.1.2
# ============================================================
import copy

params = spec.get('components', {}).get('parameters', {})
schemas = spec['components']['schemas']

# --- R2-m-1: Fix stray space/period in info.description ---
desc = spec['info']['description']
if 'capabilities . These' in desc:
    spec['info']['description'] = desc.replace('capabilities . These', 'capabilities. These')
    print("R2-m-1: Fixed stray space in info.description")

# --- R2-m-2: Rename ambiguous query param 'types' → 'transactionTypes' ---
if 'TransactionTypes' in params:
    tp = params['TransactionTypes']
    if tp.get('name') == 'types':
        tp['name'] = 'transactionTypes'
        tp['description'] = 'Filter by one or more transaction type codes (comma-separated). Values correspond to the transactionType field on each Transaction object.'
        print("R2-m-2: Renamed query param 'types' → 'transactionTypes' with clarified description")

# --- R2-m-3: Standardize PartyRole enum from PascalCase to SCREAMING_SNAKE_CASE ---
if 'PartyRole' in params:
    pr_schema = params['PartyRole']['schema']
    pascal_to_snake = {
        'Owner': 'OWNER',
        'JointOwner': 'JOINT_OWNER',
        'Annuitant': 'ANNUITANT',
        'JointAnnuitant': 'JOINT_ANNUITANT',
        'Payee': 'PAYEE',
        'Payor': 'PAYOR',
        'PrimaryBeneficiary': 'PRIMARY_BENEFICIARY',
        'ContingentBeneficiary': 'CONTINGENT_BENEFICIARY',
        'Agent': 'AGENT',
        'Trustee': 'TRUSTEE',
    }
    old_enum = pr_schema.get('enum', [])
    new_enum = [pascal_to_snake.get(v, v) for v in old_enum]
    pr_schema['enum'] = new_enum
    print(f"R2-m-3: Standardized PartyRole enum to SCREAMING_SNAKE_CASE ({len(new_enum)} values)")

# --- R2-m-4: Fix NOTAPPROVED → NOT_APPROVED ---
if 'PartyStatus' in params:
    ps_schema = params['PartyStatus']['schema']
    ps_enum = ps_schema.get('enum', [])
    if 'NOTAPPROVED' in ps_enum:
        idx = ps_enum.index('NOTAPPROVED')
        ps_enum[idx] = 'NOT_APPROVED'
        print("R2-m-4: Fixed NOTAPPROVED → NOT_APPROVED in PartyStatus enum")

# --- R2-m-5: Already fixed (no duplicates in issueCountry) ---
print("R2-m-5: ✅ Already fixed (issueCountry enum has no duplicates, boolean False → 'NO')")

# --- R2-m-6: Already fixed (all classification fields have const/enum/pattern) ---
print("R2-m-6: ✅ Already fixed (all classification fields are constrained)")

# --- R2-m-7: Rename participantParticipantRoleCode → participantRoleCode ---
if 'Rider' in schemas:
    rider_props = schemas['Rider'].get('properties', {})
    if 'participantParticipantRoleCode' in rider_props:
        rider_props['participantRoleCode'] = rider_props.pop('participantParticipantRoleCode')
        # Also fix required array if present
        rider_req = schemas['Rider'].get('required', [])
        if 'participantParticipantRoleCode' in rider_req:
            idx = rider_req.index('participantParticipantRoleCode')
            rider_req[idx] = 'participantRoleCode'
        print("R2-m-7: Renamed participantParticipantRoleCode → participantRoleCode in Rider")

# --- R2-m-8: Constrain unconstrained numeric-string fields ---
# UnderlyingAssets.transactionId has no pattern or maxLength
if 'UnderlyingAssets' in schemas:
    ua_props = schemas['UnderlyingAssets'].get('properties', {})
    if 'transactionId' in ua_props:
        tid = ua_props['transactionId']
        if 'maxLength' not in tid and 'pattern' not in tid:
            tid['maxLength'] = 100
            tid['pattern'] = '^[A-Za-z0-9._-]{1,100}$'
            print("R2-m-8: Constrained UnderlyingAssets.transactionId with pattern + maxLength")

print("\n✅ All Round 2 minor fixes applied")

In [0]:
# ============================================================
# Apply v1.1.3 Fixes — DD alignment + Round 4 items
# ============================================================
import json

params = spec['components']['parameters']
schemas = spec['components']['schemas']

# --- DD-1: Revert PartyRole enum to PascalCase (match Data Dictionary) ---
# The DD defines these as PascalCase; YAML should match DD as source of truth.
if 'PartyRole' in params:
    params['PartyRole']['schema']['enum'] = [
        'Owner', 'JointOwner', 'Annuitant', 'JointAnnuitant',
        'Payee', 'Payor', 'PrimaryBeneficiary', 'ContingentBeneficiary',
        'Agent', 'Trustee'
    ]
    print("DD-1: Reverted PartyRole enum to PascalCase (DD alignment)")

# --- DD-2: Revert PartyStatus to match DD (NOTAPPROVED) ---
if 'PartyStatus' in params:
    params['PartyStatus']['schema']['enum'] = ['APPROVED', 'NOTAPPROVED', 'PROCESSED']
    params['PartyStatus']['description'] = (
        'Filter parties by status. NOTAPPROVED is the canonical DD token '
        '(not split as NOT_APPROVED).'
    )
    print("DD-2: Reverted PartyStatus enum to DD canonical form (NOTAPPROVED)")

# --- R4-M-1: Remove UnderlyingAssets.transactionId (echoes query param) ---
# IRI DFA rule: response body is the resource, not an echo of request parameters.
# Precedent: C-2 removed policyNumber/associatedFirmId for same reason.
if 'UnderlyingAssets' in schemas:
    ua_props = schemas['UnderlyingAssets'].get('properties', {})
    if 'transactionId' in ua_props:
        del ua_props['transactionId']
        # Also remove from required if present
        ua_req = schemas['UnderlyingAssets'].get('required', [])
        if 'transactionId' in ua_req:
            ua_req.remove('transactionId')
        print("R4-M-1: Removed UnderlyingAssets.transactionId (query param echo)")

# --- R4-m-1: Fix Rider example payloads (old field name) ---
def fix_example_field_rename(d, old_name, new_name, path=""):
    fixes = 0
    if isinstance(d, dict):
        if old_name in d:
            d[new_name] = d.pop(old_name)
            fixes += 1
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                fixes += fix_example_field_rename(v, old_name, new_name, f"{path}.{k}")
    elif isinstance(d, list):
        for item in d:
            fixes += fix_example_field_rename(item, old_name, new_name, path)
    return fixes

# Fix in response examples
responses = spec.get('components', {}).get('responses', {})
ex_fixes = fix_example_field_rename(responses, 'participantParticipantRoleCode', 'participantRoleCode')
print(f"R4-m-1: Fixed {ex_fixes} example occurrences (participantParticipantRoleCode → participantRoleCode)")

# --- R4-m-3: Add maxLength to pattern-only classification fields ---
# This addresses the 'unconstrained classification' concern while preserving patterns.
classification_fields = {
    'riderSubType': 50,
    'arrSubType': 50,
    'additionalRiderClassification': 100,
    'participantRoleCode': 50,
}
class_fixes = 0
def constrain_classification_fields(d, targets):
    global class_fixes
    if isinstance(d, dict):
        for key, val in d.items():
            if key in targets and isinstance(val, dict) and val.get('type') == 'string':
                if 'maxLength' not in val:
                    val['maxLength'] = targets[key]
                    class_fixes += 1
            elif isinstance(val, (dict, list)):
                constrain_classification_fields(val, targets)
    elif isinstance(d, list):
        for item in d:
            constrain_classification_fields(item, targets)

constrain_classification_fields(schemas, classification_fields)
print(f"R4-m-3: Added maxLength to {class_fixes} classification pattern fields")

# --- R4-m-4: Add descriptions to numeric-string fields explaining string typing ---
numeric_string_docs = {
    'numberOfPayments': 'Total payment count. String type retained for carrier-specific leading-zero semantics.',
    'subAccountTerm': 'Sub-account term period. String type retained for carrier code compatibility.',
    'termPeriod': 'Term period value. String type retained for carrier-specific encoding.',
    'indexOptionPeriod': 'Index option period. String type retained for carrier code compatibility.',
    'frequencyCode': 'Frequency code value. String type retained for carrier-specific code semantics.',
}
def add_numeric_string_docs(d, docs):
    fixes = 0
    if isinstance(d, dict):
        for key, val in d.items():
            if key in docs and isinstance(val, dict) and val.get('type') == 'string':
                if not val.get('description'):
                    val['description'] = docs[key]
                    fixes += 1
            elif isinstance(val, (dict, list)):
                fixes += add_numeric_string_docs(val, docs)
    elif isinstance(d, list):
        for item in d:
            fixes += add_numeric_string_docs(item, docs)
    return fixes

doc_fixes = add_numeric_string_docs(schemas, numeric_string_docs)
print(f"R4-m-4: Documented {doc_fixes} numeric-string fields (rationale for string type)")

# --- R4-m-2: issueCountry — confirmed 0 duplicates. No action needed. ---
print("R4-m-2: ✅ Confirmed 0 duplicates in issueCountry (reviewer false positive)")

print(f"\n✅ All v1.1.3 fixes applied. Schema count: {len(schemas)}")

In [0]:
# ============================================================
# Version up to 1.1.3 and save
# ============================================================

spec['info']['version'] = '1.1.3'

# Replace any existing changelog
existing_desc = spec['info'].get('description', '')
if '---\n## Changelog' in existing_desc:
    existing_desc = existing_desc[:existing_desc.index('---\n## Changelog')].rstrip()

changelog = """\n\n---\n## Changelog v1.1.3\n### Critical Fixes (from v1.1.0 review)\n- C-1: Removed 28+ orphan/shadow schemas\n- C-2: Removed policyNumber/associatedFirmId from all response body schemas\n- C-3: Cleaned leaked parameter-echo block from TransactionsList\n- C-4: Fixed TaxWithholdingInstruction if/then guard\n- C-6: Constrained npn to pattern ^[0-9]{1,10}$\n\n### Moderate Fixes\n- M-1: Fixed required array reference after administrativeTransaction rename\n- M-2: Updated example keys to match schema renames\n- M-3: Moved pagination fields to SystematicProgramsList; added pagination to PolicyProducers\n- M-4: Removed npn from ProducerExternalIds.required (read API)\n- M-5: Removed stray partyStatus/partyRole from Party; relaxed required\n- M-5b: Fixed issueCountry enum (YAML boolean false -> 'NO'); deduplicated\n- M-6: Added pattern ^[0-9]{9}$ to all taxId fields\n- M-7: Added conditional field descriptions to SystematicProgram\n- M-9: Fixed required reference incomePayments -> incomePayment\n- M-12: Added minimum/maximum to riderBenefitPercent\n- M-13: Changed numOfModalOccurrences to integer\n- M-14: Renamed singular incomePayments -> incomePayment\n\n### Minor Fixes (v1.1.1)\n- m-2: Removed correlationId from Error body (header-only)\n- m-13: Renamed administrativeTransaction -> administrativeTransactions\n- DD-1: Tightened firstName/middleName/lastName maxLength to 100 (DD alignment)\n\n### Round 2 Minor Fixes (v1.1.2)\n- R2-m-1: Fixed stray space/period in info.description\n- R2-m-2: Renamed ambiguous query param 'types' -> 'transactionTypes'\n- R2-m-7: Renamed participantParticipantRoleCode -> participantRoleCode in Rider\n\n### Round 4 Fixes (v1.1.3)\n- DD-1: Reverted PartyRole enum to PascalCase (DD source-of-truth alignment)\n- DD-2: Reverted PartyStatus to DD canonical form (NOTAPPROVED); added clarifying description\n- R4-M-1: Removed UnderlyingAssets.transactionId (query-param echo; IRI DFA compliance)\n- R4-m-1: Fixed Rider example payloads to use renamed participantRoleCode field\n- R4-m-3: Added maxLength constraints to classification pattern fields\n- R4-m-4: Documented rationale for numeric-string typed fields"""
spec['info']['description'] = existing_desc + changelog

output_yaml = yaml.dump(spec, default_flow_style=False, sort_keys=False, allow_unicode=True, width=120)

output_path = '/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output/PolicyInquiry_v1.1.3.yml'
with open(output_path, 'w') as f:
    f.write(output_yaml)

print(f"\u2705 Saved Policy Inquiry v1.1.3")
print(f"   Path: {output_path}")
print(f"   Size: {len(output_yaml):,} chars, {output_yaml.count(chr(10))+1} lines")
print(f"   Version: {spec['info']['version']}")
print(f"   Schemas: {len(spec['components']['schemas'])}")

In [0]:
# ============================================================
# Iterative Review-Fix Loop
# Target: 100/100 | Acceptable: 95+
# Max iterations: 3 (hard cap)
# ============================================================
import re, json, yaml, time

MAX_ITERATIONS = 3
TARGET_SCORE = 100
MIN_ACCEPTABLE = 95

def load_spec_from_file(version):
    """Load spec YAML from disk."""
    path = f'/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output/PolicyInquiry_v{version}.yml'
    with open(path) as f:
        content = f.read()
    return content, path

def parse_score(review_text):
    """Extract total score from review output."""
    patterns = [
        r'\|\s*\*\*Total\*\*\s*\|\s*\*\*100\*\*\s*\|\s*\*\*(\d+)\*\*',
        r'\|\s*Total\s*\|\s*100\s*\|\s*(\d+)\s*\|',
        r'Total[^|]*\|[^|]*\|[^|]*(\d{2,3})',
        r'(\d{2,3})\s*/\s*100',
        r'score[:\s]*(\d{2,3})',
    ]
    for pattern in patterns:
        match = re.search(pattern, review_text)
        if match:
            score = int(match.group(1))
            if 0 <= score <= 100:
                return score
    return None

def extract_open_issues(review_text):
    """Extract unfixed issues from review output."""
    issues = []
    for line in review_text.split('\n'):
        if '❌' in line or '🆕' in line:
            issues.append(line.strip())
    return issues

def run_review(spec_yaml_content, version):
    """Run the review agent and return the output."""
    review_prompt = f"""Review this IRI Digital-First OpenAPI 3.1 YAML specification for Policy Inquiry v{version}.

This is a REVISION with all prior findings addressed. Perform a full review including:
- Structural validation (OpenAPI 3.1 compliance)
- Style guide enforcement (IRI conventions)
- Cross-spec consistency against published IRI specs
- Data Dictionary alignment

Also fetch and cross-check the Data Dictionary:
- Data Dictionary URL: {dd_url}

Provide:
1. The full governance scorecard with category scores (Total must be out of 100)
2. A list of ALL findings (Critical, Moderate, Minor) with evidence
3. For each finding that costs points, explain exactly what to fix

Here is the spec:

```yaml
{spec_yaml_content}
```"""
    result = agent(review_prompt)
    return str(result)

def generate_fixes(spec_dict, review_text, open_issues):
    """Ask agent to generate Python fix code for remaining issues."""
    issues_text = '\n'.join(open_issues[:15])
    
    fix_prompt = f"""You are an OpenAPI spec fixer. Given these remaining review findings, generate Python code that modifies the `spec` dict (already loaded in memory) to fix them.

REMAINING ISSUES:
{issues_text}

FULL REVIEW CONTEXT (last 3000 chars):
{review_text[-3000:]}

RULES:
- The spec is a Python dict named `spec` already in memory
- `spec['components']['schemas']` contains all schemas
- `spec['components']['parameters']` contains all parameters  
- `spec['paths']` contains all path operations
- Only output executable Python code (no markdown, no explanation)
- Do NOT redefine spec or import yaml
- Do NOT save the file (that happens separately)
- Print what you fixed
- If an issue is DD-level or requires working group decision, print a skip message
- Be conservative: don't break things that work

Generate ONLY the Python fix code:"""
    
    result = agent(fix_prompt)
    return str(result)

def apply_fix_code(fix_code_raw):
    """Extract and execute Python code from agent response."""
    code_blocks = re.findall(r'```python\n(.*?)```', fix_code_raw, re.DOTALL)
    if code_blocks:
        code = '\n'.join(code_blocks)
    else:
        lines = []
        for line in fix_code_raw.split('\n'):
            stripped = line.strip()
            if stripped and not stripped.startswith('#') and not stripped.startswith('```'):
                if any(stripped.startswith(kw) for kw in ['if ', 'for ', 'def ', 'spec', 'params', 'schemas', 'print', 'del ', 'import ', 'try', 'except', '    ']):
                    lines.append(line)
                elif '=' in stripped or '(' in stripped:
                    lines.append(line)
        code = '\n'.join(lines)
    
    if not code.strip():
        print("  ⚠️ No executable fix code extracted")
        return False
    
    try:
        exec(code, globals())
        return True
    except Exception as e:
        print(f"  ⚠️ Fix code execution failed: {e}")
        return False

def save_spec(spec_dict, version):
    """Save spec to disk with updated changelog."""
    spec_dict['info']['version'] = version
    output_yaml = yaml.dump(spec_dict, default_flow_style=False, sort_keys=False, allow_unicode=True, width=120)
    path = f'/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output/PolicyInquiry_v{version}.yml'
    with open(path, 'w') as f:
        f.write(output_yaml)
    return path, len(output_yaml)

# ============================================================
# MAIN LOOP
# ============================================================
current_version = '1.1.3'
best_score = 0
iteration = 0
results_log = []

print(f"{'='*60}")
print(f"ITERATIVE REVIEW-FIX LOOP")
print(f"Target: {TARGET_SCORE}/100 | Min acceptable: {MIN_ACCEPTABLE}/100")
print(f"Max iterations: {MAX_ITERATIONS} (hard cap)")
print(f"Starting version: v{current_version}")
print(f"{'='*60}\n")

while iteration < MAX_ITERATIONS:
    iteration += 1
    print(f"\n{'─'*50}")
    print(f"ITERATION {iteration}/{MAX_ITERATIONS}")
    print(f"{'─'*50}")
    
    # 1. Load current spec
    spec_yaml_content, spec_file_path = load_spec_from_file(current_version)
    print(f"  📄 Loaded v{current_version} ({len(spec_yaml_content):,} chars)")
    
    # 2. Run review
    print(f"  🔍 Running review agent...")
    review_output = run_review(spec_yaml_content, current_version)
    
    # 3. Parse score
    score = parse_score(review_output)
    if score is None:
        print(f"  ⚠️ Could not parse score from review output")
        print(f"  Last 500 chars: {review_output[-500:]}")
        results_log.append({'iteration': iteration, 'version': current_version, 'score': '?', 'status': 'parse_error'})
        break
    
    best_score = max(best_score, score)
    print(f"  📊 Score: {score}/100 (best: {best_score}/100)")
    
    # 4. Check if target met
    if score >= TARGET_SCORE:
        print(f"\n  🎯 TARGET ACHIEVED: {score}/100")
        results_log.append({'iteration': iteration, 'version': current_version, 'score': score, 'status': 'target_met'})
        break
    
    # 5. If this is the LAST iteration, accept 95+ or fail
    if iteration >= MAX_ITERATIONS:
        if score >= MIN_ACCEPTABLE:
            print(f"\n  ✅ Acceptable score {score}/100 after {iteration} iterations (min: {MIN_ACCEPTABLE})")
            results_log.append({'iteration': iteration, 'version': current_version, 'score': score, 'status': 'acceptable'})
        else:
            print(f"\n  ❌ Below threshold {score}/100 after {iteration} iterations — needs manual review")
            results_log.append({'iteration': iteration, 'version': current_version, 'score': score, 'status': 'below_threshold'})
        break
    
    # 6. Extract issues to fix
    open_issues = extract_open_issues(review_output)
    print(f"  📋 Open issues: {len(open_issues)}")
    for issue in open_issues[:5]:
        print(f"     • {issue[:120]}")
    
    if not open_issues:
        print(f"  ⚠️ Score < 100 but no extractable issues (soft concerns only). Stopping.")
        results_log.append({'iteration': iteration, 'version': current_version, 'score': score, 'status': 'no_actionable_issues'})
        break
    
    # 7. Generate fixes
    print(f"  🔧 Generating fixes...")
    fix_output = generate_fixes(spec, review_output, open_issues)
    
    # 8. Apply fixes
    print(f"  ⚙️ Applying fixes...")
    success = apply_fix_code(fix_output)
    
    if not success:
        print(f"  ⚠️ Fix application failed. Stopping.")
        results_log.append({'iteration': iteration, 'version': current_version, 'score': score, 'status': 'fix_failed'})
        break
    
    # 9. Bump version and save
    version_parts = current_version.split('.')
    version_parts[-1] = str(int(version_parts[-1]) + 1)
    new_version = '.'.join(version_parts)
    
    path, size = save_spec(spec, new_version)
    current_version = new_version
    print(f"  💾 Saved v{new_version} ({size:,} chars)")
    
    results_log.append({'iteration': iteration, 'version': new_version, 'score': score, 'status': 'fixed_and_saved'})

# ============================================================
# FINAL SUMMARY
# ============================================================
print(f"\n\n{'='*60}")
print(f"FINAL RESULTS")
print(f"{'='*60}")
print(f"  Best score achieved: {best_score}/100")
print(f"  Final version: v{current_version}")
print(f"  Iterations used: {iteration}/{MAX_ITERATIONS}")
print(f"\n  Iteration log:")
for entry in results_log:
    print(f"    [{entry['iteration']}] v{entry['version']} → {entry['score']}/100 ({entry['status']})")

if best_score >= TARGET_SCORE:
    print(f"\n  🎯 PERFECT SCORE ACHIEVED")
elif best_score >= MIN_ACCEPTABLE:
    print(f"\n  ✅ ACCEPTABLE — publication-ready at {best_score}/100")
else:
    print(f"\n  ❌ BELOW THRESHOLD — needs manual review")